<a href="https://colab.research.google.com/github/Myria255/BOOTCAMP-TTA/blob/main/IMDB_Classification_Dense_Colab.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# Classification des critiques IMDB avec un réseau de neurones

Ce notebook réalise une **classification binaire** de 50 000 critiques de films :

- `1` : critique positive ;
- `0` : critique négative.

Étapes : chargement, encodage one-hot, séparation entraînement/validation/test, création du modèle Dense, entraînement pendant 20 époques, analyse du surapprentissage, réentraînement et évaluation finale.

Cette première cellule de code initialise les bibliothèques nécessaires comme `numpy` pour les opérations numériques, `matplotlib.pyplot` pour la visualisation, et `tensorflow`/`keras` pour la construction et l'entraînement du modèle de réseau de neurones.

Le jeu de données IMDB est chargé à l'aide de `keras.datasets.imdb.load_data()`. L'argument `num_words=10_000` indique que seules les 10 000 mots les plus fréquents seront conservés, ce qui aide à réduire la taille du vocabulaire et la complexité du modèle. Les critiques sont déjà encodées sous forme de séquences d'entiers, où chaque entier représente un mot spécifique.

Les affichages suivants donnent un aperçu des données chargées : le nombre de critiques dans les ensembles d'entraînement et de test, un exemple de critique encodée (les premiers 20 mots), son étiquette (1 pour positif, 0 pour négatif), et l'indice maximal des mots présents, confirmant qu'il est bien inférieur à `num_words`.

In [1]:
# 1. Importation des bibliothèques et chargement des données
import numpy as np
import matplotlib.pyplot as plt
import tensorflow as tf
from tensorflow import keras
from tensorflow.keras import layers

# Reproductibilité
np.random.seed(42)
tf.random.set_seed(42)

num_words = 10_000

(train_data, train_labels), (test_data, test_labels) = \
    keras.datasets.imdb.load_data(num_words=num_words)

print("Nombre de critiques d'entraînement :", len(train_data))
print("Nombre de critiques de test :", len(test_data))
print("Exemple de critique encodée :", train_data[0][:20])
print("Étiquette de l'exemple :", train_labels[0])
print("Indice maximal rencontré :", max(max(sequence) for sequence in train_data))

17464789/17464789 ━━━━━━━━━━━━━━━━━━━━ 0s 0us/step
Nombre de critiques d'entraînement : 25000
Nombre de critiques de test : 25000
Exemple de critique encodée : [1, 14, 22, 16, 43, 530, 973, 1622, 1385, 65, 458, 4468, 66, 3941, 4, 173, 36, 256, 5, 25]
Étiquette de l'exemple : 1
Indice maximal rencontré : 9999


Cette cellule prépare les données pour l'entraînement du réseau de neurones.

La fonction `vectorize_sequences` convertit les séquences d'entiers (qui représentent les critiques) en vecteurs binaires (one-hot encoding). Pour chaque critique, un vecteur de taille 10 000 (le nombre de mots conservés) est créé. Si un mot est présent dans la critique, la position correspondante dans le vecteur est définie à `1.0`, sinon elle reste à `0.0`. Cela permet au réseau de neurones de traiter les données textuelles.

Les étiquettes (`train_labels`, `test_labels`) sont converties en tableaux `numpy` de type `float32`.

Enfin, les données d'entraînement sont divisées en un ensemble d'entraînement partiel (`partial_x_train`, `partial_y_train`) et un ensemble de validation (`x_val`, `y_val`). L'ensemble de validation est utilisé pour surveiller la performance du modèle pendant l'entraînement et détecter le surapprentissage, tandis que l'ensemble de test (`x_test`, `y_test`) est conservé pour l'évaluation finale du modèle, après que son entraînement est terminé.

In [2]:
# 2. Vectorisation one-hot et séparation des données
def vectorize_sequences(sequences, dimension=10_000):
    """Transforme chaque séquence d'indices en vecteur binaire."""
    results = np.zeros((len(sequences), dimension), dtype=np.float32)
    for i, sequence in enumerate(sequences):
        results[i, sequence] = 1.0
    return results

x_train = vectorize_sequences(train_data, num_words)
x_test = vectorize_sequences(test_data, num_words)

y_train = np.asarray(train_labels, dtype=np.float32)
y_test = np.asarray(test_labels, dtype=np.float32)

# 10 000 observations pour la validation et 15 000 pour l'entraînement initial
x_val = x_train[:10_000]
partial_x_train = x_train[10_000:]
y_val = y_train[:10_000]
partial_y_train = y_train[10_000:]

print("Entraînement :", partial_x_train.shape, partial_y_train.shape)
print("Validation   :", x_val.shape, y_val.shape)
print("Test         :", x_test.shape, y_test.shape)

Entraînement : (15000, 10000) (15000,)
Validation   : (10000, 10000) (10000,)
Test         : (25000, 10000) (25000,)


Cette cellule définit l'architecture du réseau de neurones et le compile.

La fonction `create_model` construit un modèle `Sequential` de Keras, ce qui signifie que les couches sont empilées les unes après les autres. Le modèle est composé de :
- Une couche `Input` qui spécifie la forme des données d'entrée (`num_words`,), c'est-à-dire un vecteur de 10 000 éléments.
- Deux couches `Dense` avec 16 unités chacune et une fonction d'activation `relu` (Rectified Linear Unit). Ces couches sont entièrement connectées, ce qui signifie que chaque neurone d'une couche est connecté à chaque neurone de la couche précédente.
- Une dernière couche `Dense` avec 1 unité et une fonction d'activation `sigmoid`. La fonction sigmoïde est utilisée pour la classification binaire car elle produit une sortie comprise entre 0 et 1, qui peut être interprétée comme une probabilité.

Le modèle est ensuite `compilé` :
- L'optimiseur `RMSprop` est choisi pour ajuster les poids du réseau pendant l'entraînement.
- La fonction de perte `binary_crossentropy` est utilisée car il s'agit d'un problème de classification binaire.
- La métrique `accuracy` est surveillée pour évaluer la performance du modèle pendant l'entraînement et la validation.

In [3]:
# 3. Construction et compilation du modèle
def create_model():
    model = keras.Sequential([
        layers.Input(shape=(num_words,)),
        layers.Dense(16, activation="relu"),
        layers.Dense(16, activation="relu"),
        layers.Dense(1, activation="sigmoid")
    ])

    model.compile(
        optimizer=keras.optimizers.RMSprop(),
        loss="binary_crossentropy",
        metrics=["accuracy"]
    )
    return model

model = create_model()
model.summary()

Model: "sequential"

┏━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━┓
┃ Layer (type)                    ┃ Output Shape           ┃       Param # ┃
┡━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━┩
│ dense (Dense)                   │ (None, 16)             │       160,016 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dense_1 (Dense)                 │ (None, 16)             │           272 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dense_2 (Dense)                 │ (None, 1)              │            17 │
└─────────────────────────────────┴────────────────────────┴───────────────┘

 Total params: 160,305 (626.19 KB)

 Trainable params: 160,305 (626.19 KB)

 Non-trainable params: 0 (0.00 B)

Cette cellule lance l'entraînement initial du modèle.

La méthode `model.fit()` entraîne le modèle sur les données fournies. Voici les arguments clés :
- `partial_x_train` et `partial_y_train` : Les données et les étiquettes utilisées pour l'entraînement.
- `epochs=20` : Le modèle effectuera 20 passages complets sur l'ensemble des données d'entraînement.
- `batch_size=512` : Les données d'entraînement sont divisées en lots de 512 échantillons. Le modèle met à jour ses poids après chaque lot.
- `validation_data=(x_val, y_val)` : Pendant l'entraînement, le modèle évaluera sa performance sur cet ensemble de validation à la fin de chaque époque. Cela permet de suivre le surapprentissage.
- `verbose=1` : Affiche la progression de l'entraînement pour chaque époque.

L'objet `history` contient un dictionnaire des métriques enregistrées (`loss`, `accuracy`, `val_loss`, `val_accuracy`) pour chaque époque, ce qui est essentiel pour l'analyse du surapprentissage.

In [4]:
# 4. Entraînement initial pendant 20 époques
history = model.fit(
    partial_x_train,
    partial_y_train,
    epochs=20,
    batch_size=512,
    validation_data=(x_val, y_val),
    verbose=1
)

history_dict = history.history
print("Métriques enregistrées :", history_dict.keys())

Epoch 1/20
30/30 ━━━━━━━━━━━━━━━━━━━━ 2s 51ms/step - accuracy: 0.7742 - loss: 0.5495 - val_accuracy: 0.8567 - val_loss: 0.4247
Epoch 2/20
30/30 ━━━━━━━━━━━━━━━━━━━━ 1s 28ms/step - accuracy: 0.8875 - loss: 0.3467 - val_accuracy: 0.8794 - val_loss: 0.3253
Epoch 3/20
30/30 ━━━━━━━━━━━━━━━━━━━━ 1s 18ms/step - accuracy: 0.9152 - loss: 0.2543 - val_accuracy: 0.8851 - val_loss: 0.2914
Epoch 4/20
30/30 ━━━━━━━━━━━━━━━━━━━━ 1s 24ms/step - accuracy: 0.9286 - loss: 0.2057 - val_accuracy: 0.8883 - val_loss: 0.2773
Epoch 5/20
30/30 ━━━━━━━━━━━━━━━━━━━━ 1s 25ms/step - accuracy: 0.9414 - loss: 0.1723 - val_accuracy: 0.8876 - val_loss: 0.2796
Epoch 6/20
30/30 ━━━━━━━━━━━━━━━━━━━━ 1s 28ms/step - accuracy: 0.9543 - loss: 0.1442 - val_accuracy: 0.8851 - val_loss: 0.2850
Epoch 7/20
30/30 ━━━━━━━━━━━━━━━━━━━━ 1s 24ms/step - accuracy: 0.9573 - loss: 0.1294 - val_accuracy: 0.8852 - val_loss: 0.2951
Epoch 8/20
30/30 ━━━━━━━━━━━━━━━━━━━━ 1s 18ms/step - accuracy: 0.9685 - loss: 0.1068 - val_accuracy: 0.8845 - v

Cette cellule génère des graphiques pour visualiser les courbes de perte et de précision du modèle au fil des époques.

Deux graphiques sont créés à l'aide de `matplotlib.pyplot`:
1.  **Courbe de perte :** Affiche la perte d'entraînement (`history_dict["loss"]`) et la perte de validation (`history_dict["val_loss"]`) par époque. Un écart croissant entre la perte d'entraînement et la perte de validation indique un surapprentissage.
2.  **Courbe de précision :** Affiche la précision d'entraînement (`history_dict["accuracy"]`) et la précision de validation (`history_dict["val_accuracy"]`) par époque. De même, un écart significatif entre ces deux courbes peut signaler un surapprentissage.

Ces visualisations sont cruciales pour comprendre le comportement du modèle et déterminer le moment optimal pour arrêter l'entraînement afin d'éviter le surapprentissage.

In [ ]:
# 5. Courbes de perte et de précision
epochs = range(1, len(history_dict["loss"]) + 1)

plt.figure(figsize=(8, 5))
plt.plot(epochs, history_dict["loss"], marker="o", label="Perte entraînement")
plt.plot(epochs, history_dict["val_loss"], marker="o", label="Perte validation")
plt.title("Perte pendant l'entraînement")
plt.xlabel("Époque")
plt.ylabel("Binary cross-entropy")
plt.legend()
plt.grid(True)
plt.show()

plt.figure(figsize=(8, 5))
plt.plot(epochs, history_dict["accuracy"], marker="o", label="Précision entraînement")
plt.plot(epochs, history_dict["val_accuracy"], marker="o", label="Précision validation")
plt.title("Précision pendant l'entraînement")
plt.xlabel("Époque")
plt.ylabel("Accuracy")
plt.legend()
plt.grid(True)
plt.show()

Cette cellule analyse les résultats de l'entraînement initial pour identifier l'époque optimale et évaluer le surapprentissage.

- `best_epoch` est calculé en trouvant l'époque où la `val_loss` (perte de validation) était minimale. C'est un indicateur clé du point où le modèle généralise le mieux aux nouvelles données avant de commencer à surapprendre.
- La meilleure perte de validation (`best_val_loss`) et la précision de validation associée (`best_val_accuracy`) sont affichées.
- Un seuil de 3% (`0.03`) est utilisé pour comparer la différence entre la précision d'entraînement et la précision de validation à la dernière époque. Si cette différence est supérieure à 3%, cela suggère que le modèle surapprend significativement, car il est beaucoup plus performant sur les données d'entraînement que sur les données de validation.

In [ ]:
# 6. Recherche automatique du nombre optimal d'époques
best_epoch = int(np.argmin(history_dict["val_loss"]) + 1)
best_val_loss = history_dict["val_loss"][best_epoch - 1]
best_val_accuracy = history_dict["val_accuracy"][best_epoch - 1]

print(f"Époque optimale selon val_loss : {best_epoch}")
print(f"Meilleure perte de validation  : {best_val_loss:.4f}")
print(f"Précision de validation associée : {best_val_accuracy:.4f}")

last_gap = history_dict["accuracy"][-1] - history_dict["val_accuracy"][-1]
if last_gap > 0.03:
    print("\nAnalyse : le modèle surapprend en fin d'entraînement.")
    print("La précision d'entraînement continue d'augmenter, mais la validation progresse moins.")
else:
    print("\nAnalyse : aucun surapprentissage important n'est détecté avec ce seuil.")

Cette cellule réentraîne le modèle, mais cette fois en utilisant l'ensemble complet des données d'entraînement (`x_train`, `y_train`) et en s'arrêtant à l'époque optimale identifiée précédemment.

- `keras.backend.clear_session()`: Cette commande est utilisée pour réinitialiser l'état de Keras, ce qui est crucial pour s'assurer que le nouveau modèle (`final_model`) est entraîné à partir de zéro, sans hériter des poids ou de l'état du modèle précédent.
- `final_model = create_model()`: Un nouveau modèle avec la même architecture est créé.
- `final_model.fit()`: Le modèle est entraîné sur toutes les données d'entraînement (`x_train`, `y_train`) pour `best_epoch` époques. L'ensemble de validation n'est pas utilisé ici car son rôle était de déterminer `best_epoch` et nous voulons maintenant utiliser toutes les données d'entraînement disponibles pour un entraînement plus robuste.

In [ ]:
# 7. Réentraînement sur les 25 000 critiques d'entraînement
# On recrée un modèle neuf pour éviter de conserver les poids du premier entraînement.
keras.backend.clear_session()
final_model = create_model()

final_history = final_model.fit(
    x_train,
    y_train,
    epochs=best_epoch,
    batch_size=512,
    verbose=1
)

print(f"Modèle final entraîné pendant {best_epoch} époque(s).")

Cette dernière cellule évalue la performance du modèle final sur l'ensemble de test (`x_test`, `y_test`), qui n'a jamais été utilisé pendant l'entraînement ou la validation.

- `final_model.evaluate()`: Calcule la perte et la précision du modèle sur les données de test. Cela donne une estimation impartiale de la capacité du modèle à généraliser à des données totalement nouvelles.
- Les résultats (perte et précision) sont affichés de manière claire, y compris un pourcentage de précision.
- Une interprétation est fournie pour aider à comprendre la signification des résultats, notamment en ce qui concerne la généralisation du modèle et le surapprentissage.
- Enfin, le code montre des exemples de prédictions sur les 10 premières critiques de l'ensemble de test. Pour chaque critique, la probabilité prédite d'être positive est affichée, la prédiction binaire (0 ou 1) et l'étiquette réelle, permettant une comparaison directe et une compréhension concrète des performances du modèle.

In [ ]:
# 8. Évaluation finale sur le jeu de test et rapport des résultats
test_loss, test_accuracy = final_model.evaluate(
    x_test,
    y_test,
    batch_size=512,
    verbose=1
)

print("\n========== RÉSULTATS FINAUX ==========")
print(f"Perte sur le jeu de test    : {test_loss:.4f}")
print(f"Précision sur le jeu de test : {test_accuracy:.4f}")
print(f"Précision en pourcentage     : {test_accuracy * 100:.2f} %")

print("\nInterprétation :")
print(
    "Le modèle généralise correctement si la précision de test est proche "
    "de la meilleure précision de validation. Une précision d'entraînement "
    "nettement supérieure à celle de validation indique un surapprentissage."
)

# Quelques prédictions
probabilities = final_model.predict(x_test[:10], verbose=0).flatten()
predictions = (probabilities >= 0.5).astype(int)

print("\nExemples de prédictions :")
for i, (probability, prediction, real_label) in enumerate(
    zip(probabilities, predictions, y_test[:10]), start=1
):
    print(
        f"Critique {i:02d} | probabilité positive={probability:.3f} "
        f"| prédiction={prediction} | vraie étiquette={int(real_label)}"
    )